# Pipeline A: PC-Based Causal Discovery

This notebook implements a pure PC algorithm approach for causal discovery.
- Uses PC algorithm with configurable alpha
- Applies human priors (blacklist + whitelist)
- Uses local pandas-based execution (no Spark/Databricks)
- Outputs to timestamped artifact folder

## Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Add utils to path
sys.path.insert(0, '/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/DAG discovery algorithms')
from causal_discovery_utils import *

# Causal discovery
from causallearn.search.ConstraintBased.PC import pc

print("✓ All imports successful")

## Configuration

In [ ]:
# Pipeline configuration
PIPELINE_NAME = "PC_Based"
METRICS_CSV_PATH = "/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/Metrics data/metrics.csv"
ARTIFACTS_BASE = "/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/artifacts"

# Data parameters
MAX_RUNS_TO_PIVOT = 65

# PC Algorithm parameters
PC_ALPHA = 0.05
PC_INDEP_TEST = 'fisherz'

# Feature selection parameters
TARGET_FEATURES = 40
CORRELATION_THRESHOLD = 0.95

print(f"Pipeline A Configuration:")
print(f"  - Method: PC Algorithm Only")
print(f"  - Training runs: {MAX_RUNS_TO_PIVOT}")
print(f"  - PC Alpha: {PC_ALPHA}")
print(f"  - Independence Test: {PC_INDEP_TEST}")
print(f"  - Target Features: {TARGET_FEATURES}")
print(f"  - Metrics CSV: {METRICS_CSV_PATH}")

## Step 1: Load Metrics Data

In [ ]:
print("[Step 1] Loading metrics data...")
metrics_df = load_metrics_csv(METRICS_CSV_PATH)
print(f"Loaded {metrics_df.shape[0]} metric records")
print(f"Columns: {list(metrics_df.columns)}")
print(f"Date range: {metrics_df['date'].min()} to {metrics_df['date'].max()}")

## Step 2: Transform Metrics to Matrix

In [ ]:
print("[Step 2] Transforming metrics to matrix...")
metrics_matrix = metrics_to_matrix(metrics_df, max_runs=MAX_RUNS_TO_PIVOT, stage=None)
print(f"Matrix shape: {metrics_matrix.shape}")
print(f"Columns: {list(metrics_matrix.columns[:10])}...")  # Show first 10

## Step 3: Preprocess Metrics Matrix

In [ ]:
print("\n[Step 3] Preprocessing metrics matrix...")
scaled, preprocess_meta = preprocess_metrics_matrix(
    metrics_matrix,
    zscore=True,
    feature_sample_ratio=2.5
)
print(f"After preprocessing: {scaled.shape}")
print(f"Preprocessing metadata:")
for key, val in preprocess_meta.items():
    if not isinstance(val, dict):
        print(f"  {key}: {val}")

## Step 4: Feature Selection

In [ ]:
print("\n[Step 4] Feature selection...")
final_features, feature_selection_log = sophisticated_feature_selection(
    scaled,
    target_features=TARGET_FEATURES,
    variance_threshold=1e-6,
    correlation_threshold=CORRELATION_THRESHOLD
)

n_samples, n_features = final_features.shape
print(f"\nPC Requirements Check:")
print(f"  Samples: {n_samples}, Features: {n_features}")
print(f"  Sample-to-feature ratio: {n_samples / n_features:.2f}")

if final_features.isna().any().any():
    print("WARNING: NaN values still present!")
if n_features == 0:
    print("ERROR: No features remaining!")

## Step 5: Generate Blacklist

In [ ]:
print("\n[Step 5] Generating blacklist...")
metric_cols = final_features.columns.tolist()
HUMAN_PRIOR_BLACKLIST = generate_stage_blacklist(metric_cols)
print(f"  - Blacklist edges: {len(HUMAN_PRIOR_BLACKLIST)}")
print(f"  - Sample blacklist edges: {HUMAN_PRIOR_BLACKLIST[:5]}")

## Step 6: Run PC Algorithm

In [ ]:
print(f"\n[Step 6] Running PC Algorithm...")
print(f"  Alpha: {PC_ALPHA}")
print(f"  Independence test: {PC_INDEP_TEST}")
print(f"  Data shape: {final_features.shape}")

try:
    pc_result = pc(final_features.values, alpha=PC_ALPHA, indep_test=PC_INDEP_TEST)
    print(f"  ✓ PC algorithm completed")
except Exception as e:
    print(f"  ERROR: PC algorithm failed: {e}")
    pc_result = None

## Step 7: Extract Edges from PC Result

In [ ]:
print(f"\n[Step 7] Extracting edges...")

raw_pc_edges = []
if pc_result is not None and hasattr(pc_result, 'G'):
    G_matrix = pc_result.G
    col_names = final_features.columns.tolist()
    
    for i in range(len(col_names)):
        for j in range(len(col_names)):
            if i != j and G_matrix[i, j] != 0:
                # PC returns adjacency matrix
                from_col = col_names[i]
                to_col = col_names[j]
                raw_pc_edges.append({
                    'from': from_col,
                    'to': to_col,
                    'edge_type': 'directed',
                    'weight': 1.0,
                    'source': 'pc_raw'
                })

print(f"PC discovered {len(raw_pc_edges)} edges")
if len(raw_pc_edges) == 0:
    print("WARNING: No edges discovered by PC algorithm!")

## Step 8: Apply Blacklist Filtering

In [ ]:
print(f"\n[Step 8] Applying blacklist filtering...")

selected_feature_set = set(final_features.columns)
filtered_blacklist = [
    (a, b) for a, b in HUMAN_PRIOR_BLACKLIST 
    if a in selected_feature_set and b in selected_feature_set
]
blacklist_set = set(filtered_blacklist)

print(f"Applicable blacklist edges: {len(filtered_blacklist)}")

# Filter edges based on blacklist
filtered_edges = []
removed_edges = []

for edge in raw_pc_edges:
    edge_tuple = (edge['from'], edge['to'])
    if edge_tuple not in blacklist_set:
        filtered_edges.append(edge)
    else:
        removed_edges.append(edge)

print(f"After blacklist:")
print(f"  - Edges kept: {len(filtered_edges)}")
print(f"  - Edges removed: {len(removed_edges)}")

## Step 9: Visualize Raw Graph

In [ ]:
print(f"\n[Step 9] Visualizing raw graph...")
if len(raw_pc_edges) > 0:
    G_raw = visualize_skeleton(
        [(e['from'], e['to'], 'directed') for e in raw_pc_edges],
        title="Pipeline A: PC RAW Graph (Before Blacklist)"
    )
    print(f"Raw graph nodes: {G_raw.number_of_nodes()}, edges: {G_raw.number_of_edges()}")
else:
    print("No raw edges to visualize")

## Step 10: Visualize Filtered Graph

In [ ]:
print(f"\n[Step 10] Visualizing filtered graph...")
if len(filtered_edges) > 0:
    G = visualize_skeleton(
        [(e['from'], e['to'], 'directed') for e in filtered_edges],
        title="Pipeline A: PC Graph (After Blacklist/Whitelist)"
    )
    print(f"Filtered graph nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
else:
    print("No filtered edges to visualize")

## Step 11: Compute Baseline Statistics

In [ ]:
print(f"\n[Step 11] Computing baseline statistics...")
baseline_stats = compute_baseline_stats(final_features)
print(f"Computed baseline statistics for {len(baseline_stats)} metrics")

## Step 12: Build Adjacency Maps

In [ ]:
print(f"\n[Step 12] Building adjacency maps...")
upstream_map, downstream_map = build_adjacency_maps(filtered_edges, handle_undirected=False)
print(f"Upstream map: {len(upstream_map)} nodes")
print(f"Downstream map: {len(downstream_map)} nodes")

## Step 13: Export Artifacts

In [ ]:
print(f"\n[Step 13] Exporting artifacts...")

# Create timestamped artifact directory
pipeline_path = create_timestamped_artifact_dir(ARTIFACTS_BASE, PIPELINE_NAME)

print(f"\nSaving artifacts:")

# Save core artifacts
artifacts = {
    "pipeline": PIPELINE_NAME,
    "method": "pc-based",
    "data_type": "cross-sectional",
    "status": "SUCCESS" if pc_result is not None else "FAILED",
    "timestamp": datetime.now().isoformat(),
    "training_runs": MAX_RUNS_TO_PIVOT,
    "preprocess_meta": preprocess_meta,
    "feature_selection_log": feature_selection_log,
    "pc_result": {
        "method": "pc",
        "alpha": PC_ALPHA,
        "indep_test": PC_INDEP_TEST,
        "n_samples": n_samples,
        "n_features": n_features,
        "sample_feature_ratio": n_samples / n_features,
        "raw_edges_found": len(raw_pc_edges)
    },
    "edge_stats": {
        "raw_edges": len(raw_pc_edges),
        "removed_by_blacklist": len(removed_edges),
        "final_edges": len(filtered_edges),
        "edge_type": "directed",
        "orientation_method": "pc_algorithm"
    },
    "raw_pc_edges": raw_pc_edges,
    "filtered_edges": filtered_edges,
    "blacklist_filtering": {
        "blacklist_edges_applicable": len(filtered_blacklist),
        "edges_removed": [(e['from'], e['to']) for e in removed_edges]
    },
    "final_graph_stats": {
        "total_edges": len(filtered_edges),
        "nodes_with_parents": len(upstream_map),
        "nodes_with_children": len(downstream_map)
    }
}

# Save JSON artifacts
save_json_artifact(pipeline_path, 'causal_artifacts.json', artifacts)
save_json_artifact(pipeline_path, 'baseline_stats.json', baseline_stats)
save_json_artifact(pipeline_path, 'upstream_map.json', upstream_map)
save_json_artifact(pipeline_path, 'downstream_map.json', downstream_map)

# Save CSV artifacts
save_csv_artifact(pipeline_path, 'causal_metrics_matrix.csv', final_features)

if len(raw_pc_edges) > 0:
    save_csv_artifact(pipeline_path, 'pc_raw_edges.csv', raw_pc_edges)
    raw_upstream, raw_downstream = build_adjacency_maps(raw_pc_edges, handle_undirected=False)
    save_json_artifact(pipeline_path, 'raw_upstream_map.json', raw_upstream)
    save_json_artifact(pipeline_path, 'raw_downstream_map.json', raw_downstream)

if len(filtered_edges) > 0:
    save_csv_artifact(pipeline_path, 'pc_causal_edges.csv', filtered_edges)

print(f"\n" + "="*80)
print(f"✓ PIPELINE A COMPLETE — All artifacts saved")
print(f"="*80)
print(f"\nSaved to: {pipeline_path}")
print(f"\nFinal Results:")
print(f"  - RAW edges: {len(raw_pc_edges)}")
print(f"  - FILTERED edges: {len(filtered_edges)}")
print(f"  - Removed: {len(removed_edges)}")